# Module 4 lab: explicit payload contracts

Standard library only; invalid payloads must not mutate state.

## Objectives and predictions

Predict 1: is True a strict integer? Predict 2: is omitted note the same as null note? Predict 3: can owner_id in input change stored ownership?

In [ ]:
ALLOWED={"title","priority","note"}; mutations=[]
def validate(p,content_type="application/json"):
 e=[]
 if content_type!="application/json": e.append(("body","unsupported_media"))
 if not isinstance(p,dict): return [("body","object_required")]
 e += [(k,"unknown_field") for k in sorted(set(p)-ALLOWED)]
 if "title" not in p: e.append(("title","required"))
 elif not isinstance(p["title"],str) or not p["title"].strip(): e.append(("title","string_required"))
 elif len(p["title"])>40: e.append(("title","too_long"))
 if "priority" in p and (isinstance(p["priority"],bool) or not isinstance(p["priority"],int) or not 1<=p["priority"]<=3): e.append(("priority","integer_1_to_3"))
 if "note" in p and p["note"] is not None and not isinstance(p["note"],str): e.append(("note","string_or_null"))
 return e
def create(p,subject):
 e=validate(p)
 if e: return {"status":400,"body":{"error":{"code":"invalid_field","fields":e}}}
 d={"title":p["title"].strip(),"priority":p.get("priority",2),"note":p.get("note"),"owner_id":subject}; mutations.append(d)
 return {"status":201,"body":{k:d[k] for k in ("title","priority","note")}}
assert validate({"title":"x","priority":True})[0][1]=="integer_1_to_3"; assert validate({"title":"x","owner_id":"admin"})[0][1]=="unknown_field"
assert create({"title":"x","owner_id":"admin"},"alice")["status"]==400 and not mutations
ok=create({"title":" Plan ","note":None},"alice"); assert ok["status"]==201 and mutations[0]["owner_id"]=="alice"

## Prediction answers

1. Strict validation rejects True as an integer because booleans are a distinct contract type here. 2. No: omission can leave a note unchanged while null can clear it. 3. No: owner_id is unknown/protected input and must not change the domain owner.

## Omission, null, bounds, and failure paths

Input, domain, and response models are distinct. Omitted can mean leave unchanged; null can mean clear. Validate before mutation.

In [ ]:
def update_note(p,old):
 if "note" not in p: return old
 if p["note"] is None or isinstance(p["note"],str): return p["note"]
 raise ValueError("invalid")
assert update_note({}, "old")=="old" and update_note({"note":None},"old") is None
def depth(v,n=0):
 if n>3:return False
 if isinstance(v,dict):return all(depth(x,n+1) for x in v.values())
 if isinstance(v,list):return all(depth(x,n+1) for x in v)
 return True
assert not depth({"a":{"b":{"c":{"d":1}}}})
before=len(mutations)
for bad in [{},{"title":"x"*41},{"title":"x","priority":"2"}]: assert create(bad,"alice")["status"]==400
assert len(mutations)==before

## AI-style critique and TODO

record.update(payload) mass-assigns. Guided TODO: project only public response keys.

In [ ]:
ai="record.update(payload); return record"; assert "update" in ai
def public_response(r): return {k:r[k] for k in ("title","priority","note") if k in r}
assert public_response({"title":"x","owner_id":"secret"})=={"title":"x"}

## Independent challenge

Add tags, maximum three strings of length ten, and decide duplicate policy. Exit answers: create/update differ because omission and server-managed fields differ; mass assignment copies caller fields into protected fields; reject before persistence; this does not test third-party coercion.

Evidence: payload corpus, mutation count, stable field errors, model mappings, AI review, and clean run.

## Baseline reproduction: slow path
The baseline permissive update copies caller fields into storage. Characterize its danger with synthetic data before using an allow-list.

In [ ]:
baseline_record={"title":"old","owner_id":"alice","status":"open"}
incoming={"title":"new","owner_id":"admin"}
baseline_record.update(incoming)
assert baseline_record["owner_id"]=="admin"
print("Baseline mass-assignment flaw reproduced: caller changed owner.")

## Pre-edit hypothesis
Write: “If validation rejects unknown/protected keys and runs before the mutation counter changes, then invalid payloads cannot change ownership or create partial state.”

In [ ]:
pre_edit_hypothesis="strict unknown-field rejection before mutation prevents mass assignment"
assert "before mutation" in pre_edit_hypothesis

## Incremental guided implementation: parsing and shape
Check content type and top-level object first. Then validate each field, keeping errors deterministic and field-addressable.

In [ ]:
def parse_request(payload,content_type="application/json"):
 if content_type!="application/json": return [("body","unsupported_media")]
 return validate(payload)
assert parse_request({"title":"x"},"text/plain")==[("body","unsupported_media")]
assert parse_request({"title":"x"})==[]

In [ ]:
def reference_create(payload,subject):
 errors=parse_request(payload)
 if errors: return {"status":400,"body":{"error":{"fields":errors}}}
 domain={"title":payload["title"].strip(),"priority":payload.get("priority",2),"note":payload.get("note"),"owner_id":subject}
 mutations.append(domain)
 return {"status":201,"body":public_response(domain)}
before=len(mutations)
assert reference_create({"title":"ok","owner_id":"admin"},"alice")["status"]==400 and len(mutations)==before
assert reference_create({"title":" ok "},"alice")["body"]["title"]=="ok"

## Required, optional, nullable, omitted
A create title is required. Priority is optional with a default. Note is nullable. On update, omission means leave old data while null explicitly clears it.

In [ ]:
old_note="keep"
assert update_note({},old_note)=="keep"
assert update_note({"note":None},old_note) is None
assert update_note({"note":"new"},old_note)=="new"


## Positive, negative, and failure checks
Include valid data, wrong type, enum/range failure, unknown fields, unsupported media, oversized text, and structural depth. The mutation count should remain unchanged for every invalid case.

In [ ]:
before=len(mutations)
bad=[{"title":""},{"title":"x","priority":True},{"title":"x","owner_id":"admin"},{"title":"x"*41}]
for payload in bad: assert reference_create(payload,"alice")["status"]==400
assert len(mutations)==before
assert not depth({"a":{"b":{"c":{"d":1}}}})

## AI-style/broken-code critique
The AI answer silently coerces types and exposes its storage record. Do not accept it because one valid example passes.

In [ ]:
broken_validation="payload['priority']=int(payload['priority']); return storage_row"
assert "int(" in broken_validation and "storage_row" in broken_validation
print("Reject silent coercion and storage-shaped responses.")

## Guided TODO: attempt
Write a projection that copies only title, priority, and note; protected owner/status fields are excluded.

In [ ]:
def todo_public(record):
 return {key:record[key] for key in ("title","priority","note") if key in record}
assert todo_public({"title":"x","owner_id":"secret","status":"open"})=={"title":"x"}

## Reference solution
The response mapping is an allow-list. The domain mapping separately injects subject identity, so a client cannot mass-assign it.

In [ ]:
def reference_public(record):
 return {"title":record["title"],"priority":record["priority"],"note":record["note"]}
assert reference_public({"title":"x","priority":2,"note":None,"owner_id":"secret"})=={"title":"x","priority":2,"note":None}

## Independent challenge
Add tags with at most three strings of length ten and a deterministic tags-too-large error. Choose whether duplicates are permitted and document it.

In [ ]:
def valid_tags(tags):
 return isinstance(tags,list) and len(tags)<=3 and all(isinstance(t,str) and len(t)<=10 for t in tags)
assert valid_tags(["api","sql"]) and not valid_tags(["x"]*4) and not valid_tags(["long-name!!"])

## Exit questions and Answers
Required means a key must exist; optional means omission is allowed; nullable means null is allowed. Unknown/protected input must not reach mutation. Input, domain, and response models have different authority and exposure. Validation must precede business logic/persistence.

## Evidence handoff
Keep the permissive baseline, hypothesis, payload corpus, mutation counts, omission/null notes, field errors, model mappings, AI critique, TODO/reference, challenge, and fresh execution.